# ISyE 6525 HW2 — Question 4
## Edge detection and segmentation

Run this notebook from `fda/lab2/` with `Flower.jpg` in the same directory. The original colour image is used for k-means; all edge and Otsu analyses use its 8-bit grayscale conversion.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from scipy.ndimage import convolve
from sklearn.cluster import KMeans

In [ ]:
IMAGE_PATH = Path("Flower.jpg")
if not IMAGE_PATH.is_file():
    raise FileNotFoundError(f"Expected {IMAGE_PATH.resolve()}")
with Image.open(IMAGE_PATH) as source:
    flower_rgb = np.asarray(source.convert("RGB"), dtype=np.uint8)
    flower_gray = np.asarray(source.convert("L"), dtype=np.uint8)
gray_float = flower_gray.astype(np.float64)
print(f"RGB shape: {flower_rgb.shape}; grayscale shape: {flower_gray.shape}")
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(flower_rgb)
axes[0].set_title("Original colour image")
axes[1].imshow(flower_gray, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("8-bit grayscale reference")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## (a) Laplacian of Gaussian (8 pts)

For

$$
G(x,y)=\frac{1}{2\pi\sigma^2}\exp\left(-\frac{x^2+y^2}{2\sigma^2}\right),
$$

compute $\nabla^2G=\frac{\partial^2G}{\partial x^2}+\frac{\partial^2G}{\partial y^2}$ and show the derivation.

### Derivation
For fixed $\sigma>0$, write $G=G(x,y)$. The first derivative is

$$
\frac{\partial G}{\partial x}
=\frac{1}{2\pi\sigma^2}\left(-\frac{2x}{2\sigma^2}\right)\exp\left(-\frac{x^2+y^2}{2\sigma^2}\right)
=-\frac{x}{\sigma^2}G.
$$

Second partial derivative with respect to $x$:
$$
\begin{aligned}
\frac{\partial^2G}{\partial x^2}
&=-\frac{1}{\sigma^2}G-\frac{x}{\sigma^2}\frac{\partial G}{\partial x}\\
&=-\frac{1}{\sigma^2}G+\frac{x^2}{\sigma^4}G\\
&=\frac{x^2-\sigma^2}{\sigma^4}G.
\end{aligned}
$$

By symmetry,

$$
\frac{\partial^2G}{\partial y^2}=\frac{y^2-\sigma^2}{\sigma^4}G.
$$

Adding the two second partial derivatives gives

$$
\boxed{\nabla^2G(x,y)=\frac{x^2+y^2-2\sigma^2}{\sigma^4}G(x,y)
=\frac{x^2+y^2-2\sigma^2}{2\pi\sigma^6}\exp\left(-\frac{x^2+y^2}{2\sigma^2}\right).}
$$

### Construct and apply the Laplacian-of-Gaussian kernel

- Construct a $5\times5$ LoG kernel with $\sigma=1$ on $x,y\in\{-2,-1,0,1,2\}$.
- Report the kernel numerically.
- Explain why the sampled kernel should be shifted so its entries sum to zero.
- Convolve it with the grayscale image and display the result.

In [ ]:
# Sample the derived Laplacian of Gaussian, not the Gaussian itself.
sigma = 1.0
yy, xx = np.mgrid[-2:3, -2:3]
radius_sq = xx ** 2 + yy ** 2
log_kernel_raw = ((radius_sq - 2 * sigma ** 2) / (2 * np.pi * sigma ** 6)
                  * np.exp(-radius_sq / (2 * sigma ** 2)))
print("Raw 5×5 LoG kernel (sigma=1):")
print(np.array2string(log_kernel_raw, precision=6, suppress_small=True))
print(f"Raw sampled sum: {log_kernel_raw.sum():+.8f}")

# Correct truncation/discretization error so the discrete derivative kills constants.
log_kernel = log_kernel_raw - log_kernel_raw.mean()
print("Zero-sum 5×5 LoG kernel:")
print(np.array2string(log_kernel, precision=6, suppress_small=True))
print(f"Shifted sampled sum: {log_kernel.sum():+.3e}")
np.testing.assert_allclose(log_kernel.sum(), 0.0, atol=1e-12)

# Use floating point: convolving a uint8 input can truncate/wrap signed responses.
flower_log = convolve(gray_float, log_kernel, mode="reflect")
scale = np.max(np.abs(flower_log))
plt.figure(figsize=(7, 5))
plt.imshow(flower_log, cmap="gray", vmin=-scale, vmax=scale)
plt.title("Signed Laplacian-of-Gaussian response (5×5, σ=1)")
plt.colorbar(label="LoG response")
plt.axis("off")
plt.tight_layout()
plt.show()

### Zero-sum explanation

The continuous Laplacian of Gaussian integrates to zero on the infinite plane because it is a second derivative of a rapidly decaying Gaussian. Sampling it on a finite 5×5 grid generally produces a nonzero sum: truncation and discretization introduce a small constant-component response. Subtracting the sampled kernel's mean enforces \(\sum_{i,j}h_{ij}=0\), so its convolution with an exactly constant image is zero (including borders when using `mode="reflect"`). The corrected kernel therefore responds to local intensity *changes*, rather than incorrectly responding to uniform brightness. The LoG response is **signed**; a signed response image is not yet a binary zero-crossing edge map.

## (b) Sobel and Prewitt edge detection (7 pts)

For both the Sobel and Prewitt operators:

- Compute the first partial derivatives $f_x$ and $f_y$ of the grayscale image.
- Form $|\nabla f|=\sqrt{f_x^2+f_y^2}$.
- Display thresholded edge maps for at least four cutoff values.
- Describe what happens as the cutoff increases.
- State which operator gives the larger response and explain why.

### Sobel operator

Use the conventional unnormalized 3×3 Sobel derivative masks. Their central weights of 2 smooth the direction perpendicular to the derivative more strongly than Prewitt. Plot signed \(f_x\), \(f_y\), gradient magnitude, and binary edge maps using the **same absolute cutoffs** for both operators.

In [ ]:
# scipy.ndimage.convolve flips the mask. Its sign convention is immaterial
# to gradient magnitude, but using the same convention for both operators matters.
sobel_x = np.array([[-1, 0, 1],
                    [-2, 0, 2],
                    [-1, 0, 1]], dtype=float)
sobel_y = sobel_x.T
sobel_fx = convolve(gray_float, sobel_x, mode="reflect")
sobel_fy = convolve(gray_float, sobel_y, mode="reflect")
sobel_magnitude = np.hypot(sobel_fx, sobel_fy)

# Cutoffs are raw gradient units for 8-bit input and UNNORMALIZED derivative masks.
edge_cutoffs = (40, 80, 120, 160)

def show_derivatives_and_edges(fx, fy, magnitude, operator, cutoffs):
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    derivative_limit = max(float(np.abs(fx).max()), float(np.abs(fy).max()))
    for ax, values, title in zip(
        axes, (fx, fy, magnitude), ("$f_x$", "$f_y$", "$|\\nabla f|$")
    ):
        kwargs = (dict(vmin=-derivative_limit, vmax=derivative_limit)
                  if values is not magnitude else dict(vmin=0))
        ax.imshow(values, cmap="gray", **kwargs)
        ax.set_title(f"{operator}: {title}")
        ax.axis("off")
    plt.tight_layout()
    plt.show()
    fig, axes = plt.subplots(1, len(cutoffs), figsize=(4 * len(cutoffs), 4))
    for ax, cutoff in zip(axes, cutoffs):
        binary_map = magnitude >= cutoff
        ax.imshow(binary_map, cmap="gray", vmin=0, vmax=1)
        ax.set_title(f"{operator}: cutoff ≥ {cutoff}\\n{binary_map.mean():.1%} edge pixels")
        ax.axis("off")
    plt.tight_layout()
    plt.show()

show_derivatives_and_edges(sobel_fx, sobel_fy, sobel_magnitude, "Sobel", edge_cutoffs)

### Prewitt operator

Use the conventional unnormalized Prewitt masks, which weight all three samples equally in the direction perpendicular to the derivative. Display the same four cutoffs used above to make the edge maps directly comparable.

In [ ]:
prewitt_x = np.array([[-1, 0, 1],
                      [-1, 0, 1],
                      [-1, 0, 1]], dtype=float)
prewitt_y = prewitt_x.T
prewitt_fx = convolve(gray_float, prewitt_x, mode="reflect")
prewitt_fy = convolve(gray_float, prewitt_y, mode="reflect")
prewitt_magnitude = np.hypot(prewitt_fx, prewitt_fy)
show_derivatives_and_edges(prewitt_fx, prewitt_fy, prewitt_magnitude,
                           "Prewitt", edge_cutoffs)

# Quantify the comparison on THIS image rather than asserting pointwise dominance.
print(f"Mean raw gradient: Sobel={sobel_magnitude.mean():.3f}, "
      f"Prewitt={prewitt_magnitude.mean():.3f}")
print(f"Maximum raw gradient: Sobel={sobel_magnitude.max():.3f}, "
      f"Prewitt={prewitt_magnitude.max():.3f}")
print(f"Fraction of pixels with stronger Sobel response: "
      f"{np.mean(sobel_magnitude > prewitt_magnitude):.2%}")

### Edge-map comparison

Increasing a cutoff removes lower-magnitude responses, leaving fewer and typically more prominent edges; sufficiently high cutoffs can discard genuine boundaries as well as noise. With these **unnormalized** masks, Sobel generally produces a larger raw gradient magnitude on this image because the centre row/column carries weight 2 instead of 1; its weighted orthogonal smoothing also affects noise sensitivity. This is **not** a pointwise guarantee or an intrinsic superiority of Sobel: report the printed image-specific mean, maximum, and pixel fraction, and note that dividing the masks by their gains (Sobel 8, Prewitt 6 for a unit linear ramp) changes the scale comparison.

## (c) Colour segmentation with k-means (6 pts)

Treat each RGB pixel as a point in \(\mathbb{R}^3\), run k-means for \(k\in\{2,3,4,5\}\), and replace each pixel by its assigned RGB centre. Fit on **all original colour pixels**, with a fixed random seed and multiple initializations for reproducibility. Print each cluster's centre in the original 0–255 RGB scale (cluster order is arbitrary).

In [ ]:
pixel_rgb = flower_rgb.reshape(-1, 3).astype(np.float64)
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
kmeans_results = {}
for ax, k in zip(axes.flat, (2, 3, 4, 5)):
    model = KMeans(n_clusters=k, n_init=5, max_iter=300, random_state=6525)
    labels = model.fit_predict(pixel_rgb)
    centres = model.cluster_centers_
    segmented = np.clip(centres[labels], 0, 255).reshape(flower_rgb.shape).astype(np.uint8)
    kmeans_results[k] = {"centres": centres.copy(), "segmented": segmented}
    ax.imshow(segmented)
    ax.set_title(f"k-means colour segmentation: k={k}")
    ax.axis("off")
    print(f"k={k} RGB cluster centres (rows = clusters; values in [0,255]):")
    print(np.array2string(centres, precision=2, suppress_small=True))
plt.tight_layout()
plt.show()

## (d) Multilevel Otsu segmentation (4 pts)

Optimize the **global multilevel Otsu objective** over contiguous grayscale histogram classes, rather than running ordinary binary Otsu repeatedly or using k-means. For each interval of intensities \([a,b)\), let \(n_{a:b}\) be its pixel count and \(s_{a:b}\) its intensity sum. Maximizing the between-class variance is equivalent (up to constants independent of thresholds) to maximizing \(\sum_j s_j^2/n_j\). Dynamic programming finds optimal nonempty partitions into 2, 3, 4 and 5 classes in \(O(KL^2)\) for \(L=256\) gray levels. Report the resulting \(K-1\) thresholds, then display each image quantized to its class's observed mean intensity.

In [ ]:
def multilevel_otsu_thresholds(image, n_classes):
    """Return K-1 globally optimal 8-bit Otsu thresholds via histogram DP.

    Each class occupies a contiguous interval of intensities and contains
    at least one pixel. A threshold t puts intensities <= t into the lower class.
    """
    gray = np.asarray(image)
    if gray.ndim != 2 or gray.dtype != np.uint8:
        raise ValueError("Expected a 2-D uint8 grayscale image")
    if not 2 <= n_classes <= 256:
        raise ValueError("n_classes must lie in [2,256]")
    histogram = np.bincount(gray.ravel(), minlength=256).astype(np.float64)
    if np.count_nonzero(histogram) < n_classes:
        raise ValueError("Too few distinct intensities for requested classes")
    cumulative_count = np.concatenate(([0.0], np.cumsum(histogram)))
    cumulative_sum = np.concatenate(
        ([0.0], np.cumsum(histogram * np.arange(256)))
    )

    # dp[k, end]: optimal sum(s_class**2 / n_class) for k nonempty
    # contiguous classes covering integer gray values [0, end).
    dp = np.full((n_classes + 1, 257), -np.inf, dtype=float)
    previous = np.full((n_classes + 1, 257), -1, dtype=np.int32)
    dp[0, 0] = 0.0
    for k in range(1, n_classes + 1):
        for end in range(k, 257):
            start = np.arange(k - 1, end)
            count = cumulative_count[end] - cumulative_count[start]
            intensity_sum = cumulative_sum[end] - cumulative_sum[start]
            valid = (count > 0) & np.isfinite(dp[k - 1, start])
            candidate = np.full(start.shape, -np.inf, dtype=float)
            candidate[valid] = (dp[k - 1, start][valid]
                                + intensity_sum[valid] ** 2 / count[valid])
            best = int(np.argmax(candidate))
            if np.isfinite(candidate[best]):
                dp[k, end] = candidate[best]
                previous[k, end] = start[best]

    if not np.isfinite(dp[n_classes, 256]):
        raise ValueError("Unable to construct the requested nonempty classes")
    thresholds = []
    end = 256
    for k in range(n_classes, 1, -1):
        start = int(previous[k, end])
        thresholds.append(start - 1)
        end = start
    return np.asarray(thresholds[::-1], dtype=np.uint8)


fig, axes = plt.subplots(2, 2, figsize=(12, 9))
otsu_results = {}
for ax, n_classes in zip(axes.flat, (2, 3, 4, 5)):
    thresholds = multilevel_otsu_thresholds(flower_gray, n_classes)
    # Threshold t belongs to the lower class (side='left').
    labels = np.searchsorted(thresholds, flower_gray, side="left")
    mean_intensities = np.array(
        [float(flower_gray[labels == label].mean()) for label in range(n_classes)]
    )
    quantized = np.rint(mean_intensities[labels]).astype(np.uint8)
    otsu_results[n_classes] = {
        "thresholds": thresholds, "means": mean_intensities, "image": quantized
    }
    ax.imshow(quantized, cmap="gray", vmin=0, vmax=255)
    ax.set_title(f"Otsu: {n_classes} levels; thresholds={thresholds.tolist()}")
    ax.axis("off")
    print(f"{n_classes} levels: thresholds={thresholds.tolist()}; "
          f"class means={np.round(mean_intensities, 2).tolist()}")
plt.tight_layout()
plt.show()